# 从 UniPert-G2CP 提取与 PTDS 同细胞系、同药物的 LINCS 表达签名

在 Google Colab CPU 上从上到下运行。独立于 `hepg2_alignment_drive.ipynb`。

- 论文：UniPert-G2CP bridges genetic and chemical screens from molecular representation to phenotype modeling，DOI [10.1016/j.cell.2026.06.005](https://doi.org/10.1016/j.cell.2026.06.005)。
- 作者数据：[Zenodo 20355906](https://zenodo.org/records/20355906)；[复现代码](https://github.com/lynn-1998/UniPert-G2CP_reproduce)。
- 范围：作者 LINCS 五个细胞系与当前 PTDS 仅 MCF7 重合。本 notebook 提取 `MCF7_chem_ctrl_adata.h5ad`，不使用 CPJUMP1 形态学特征。
- 匹配键：细胞系 + 药物母体的标准化结构（保留立体化学）；**剂量和时间不参与筛选、不合并不同条件**。保留盐型原名、匹配依据和原始 RNA 元数据供复核。
- 论文 LINCS Methods 报告：LINCS2020 Level 5、978 个 landmark 基因，药物条件 24 h、10 µM。数据是处理后表达签名，不是单细胞原始 counts；不做 pseudobulk 或再次 log/归一化。元数据没有时间/剂量时不伪造实测字段。
- PTDS 输入只需 `01_sample_info.xlsx`，不需要 331 MB 的蛋白表达矩阵。首次运行若默认 Drive 路径不存在，会弹出上传框。
- 默认输出：`MyDrive/virtualcell_ptds_unipert_alignment_v1/`。这是个人 Drive 路径，其他成员不能直接访问。

首次运行需要网络访问 Zenodo 和 PubChem。实际匹配数量由运行结果确定；此文件不包含虚构的执行输出。


In [ ]:
%pip -q install "pandas>=2,<3" "numpy>=1.26,<3" "anndata>=0.11,<0.13" openpyxl scipy requests rdkit


## 1. 挂载 Drive 和配置
可直接修改 `PTDS_XLSX` 指向你已有的样本表。不要填 Windows 的 `C:/...` 路径。
`MCF7` 对应 Cellosaurus `CVCL_0031`。后续若加入其他蛋白组文件，需要重新核对共同细胞系。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import io, json, re, time, hashlib, zipfile, shutil, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import anndata as ad
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from rdkit import Chem, rdBase
from rdkit.Chem.MolStandardize import rdMolStandardize
from IPython.display import display

BASE = Path('/content/drive/MyDrive/virtualcell_ptds_unipert_alignment_v1')
PTDS_XLSX = Path('/content/drive/MyDrive/CrossOmics-Generalization/baseline/proteintalks/01_sample_info.xlsx')
RAW, OUT = BASE/'raw', BASE/'results'
for p in (RAW, OUT): p.mkdir(parents=True, exist_ok=True)
assert Path('/content/drive/MyDrive').is_dir(), '请先挂载 Drive'
if not PTDS_XLSX.is_file():
    from google.colab import files
    uploaded = files.upload()  # 只上传 01_sample_info.xlsx
    assert '01_sample_info.xlsx' in uploaded, '请上传 PTDS 的 01_sample_info.xlsx'
    PTDS_XLSX = RAW/'01_sample_info.xlsx'
    PTDS_XLSX.write_bytes(uploaded['01_sample_info.xlsx'])
TARGET_CELL = 'MCF7'
RECORD_ID = 20355906
session = requests.Session()
session.mount('https://', HTTPAdapter(max_retries=Retry(total=4, backoff_factor=1,
    status_forcelist=[429,500,502,503,504], allowed_methods=['GET'])))
session.headers['User-Agent'] = 'PTDS-UniPert-alignment/1.0'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024**2), b''): h.update(b)
    return h.hexdigest()
def save_csv(df, name):
    df.to_csv(OUT/name, index=False, encoding='utf-8-sig')
def cell_key(x): return re.sub(r'[^A-Z0-9]', '', str(x).upper())
print('输入:', PTDS_XLSX, '\n输出:', OUT)


## 2. 只下载 MCF7 化学扰动文件和化合物结构表
作者 `G2CP:data.zip` 约 3.25 GB，但本次两个 ZIP 成员的压缩大小总计约 58 MB。
通过 HTTP Range 请求读取目录及目标成员；若服务器不支持范围读取则明确报错，不自动改为全包下载。
ZIP 成员 CRC 由 zipfile 校验；记录提取文件 SHA-256，重复运行时复用已校验文件。


In [ ]:
class RangeReader(io.RawIOBase):
    def __init__(self, url, size):
        self.url, self.size, self.pos = url, size, 0
        self.start, self.cache = 0, b''
    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos
    def seek(self, offset, whence=0):
        self.pos = offset if whence == 0 else (self.pos if whence == 1 else self.size)+offset
        if self.pos < 0: raise ValueError('Negative seek')
        return self.pos
    def read(self, n=-1):
        n = self.size-self.pos if n < 0 else min(n, self.size-self.pos)
        if n <= 0: return b''
        if not (self.start <= self.pos and self.pos+n <= self.start+len(self.cache)):
            end=min(self.size-1,self.pos+max(n,8*1024**2)-1)
            with session.get(self.url, headers={'Range':f'bytes={self.pos}-{end}'},
                             stream=True, timeout=(30,180)) as r:
                if r.status_code != 206:
                    raise RuntimeError(f'需要 HTTP 206 Range 支持，收到 {r.status_code}')
                cr=r.headers.get('Content-Range','')
                if not cr.startswith(f'bytes {self.pos}-{end}/'):
                    raise RuntimeError(f'Range 响应范围不一致: {cr}')
                self.cache=b''.join(r.iter_content(1024**2))
                if len(self.cache) != end-self.pos+1: raise IOError('Range 响应不完整')
            self.start=self.pos
        data=self.cache[self.pos-self.start:self.pos-self.start+n]
        self.pos+=len(data)
        return data

r=session.get(f'https://zenodo.org/api/records/{RECORD_ID}', timeout=60)
r.raise_for_status()
record=r.json()
(RAW/'zenodo_record.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
archive=next(x for x in record['files'] if x['key']=='G2CP:data.zip')
members={
    'data/LINCS/MCF7/MCF7_chem_ctrl_adata.h5ad': RAW/'MCF7_chem_ctrl_adata.h5ad',
    'data/processed_CMAP_compound_info.csv': RAW/'processed_CMAP_compound_info.csv',
}
manifest_path=RAW/'extraction_manifest.json'
manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
with zipfile.ZipFile(RangeReader(archive['links']['self'], archive['size'])) as z:
    for member, dest in members.items():
        info=z.getinfo(member)
        previous=manifest.get(member,{})
        if (dest.exists() and previous.get('crc32')==info.CRC
            and previous.get('archive_checksum')==archive.get('checksum')
            and previous.get('sha256')==sha256(dest)):
            print('复用:',dest.name)
            continue
        print('提取:',member,'压缩 MB=',round(info.compress_size/1e6,2),flush=True)
        temp=dest.with_suffix(dest.suffix+'.part')
        try:
            with z.open(info) as src, temp.open('wb') as dst:
                shutil.copyfileobj(src,dst,length=1024**2)
            assert temp.stat().st_size==info.file_size
            temp.replace(dest)
        finally:
            temp.unlink(missing_ok=True)
        manifest[member]={'sha256':sha256(dest),'crc32':info.CRC,'bytes':info.file_size,
                          'archive_checksum':archive.get('checksum'),'source':archive['links']['self']}
        manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf-8')


## 3. 读取并审计真实字段
只匹配 `trt_cp` 药物签名。对照单独保存，基因扰动不纳入。
PTDS 的 `no` 仅列为对照候选，不将其强行解释为 LINCS 的相同对照。


In [ ]:
ptds=pd.read_excel(PTDS_XLSX, dtype={'Sample_ID':str})
required={'Sample_ID','Cell_Line','Drug_Type','Pert_Name1','Pert_Name2'}
assert required <= set(ptds.columns), f'PTDS 缺少字段: {required-set(ptds.columns)}'
ptds['cell_key']=ptds['Cell_Line'].map(cell_key)
single=ptds.loc[ptds.Drug_Type.eq('single') & ptds.cell_key.eq(TARGET_CELL)].copy()
assert len(single), 'PTDS 没有 MCF7 单药类型记录'
control_protein=single.Pert_Name1.fillna('').astype(str).str.strip().str.lower().isin(['no','dmso','control','ctrl','vehicle',''])
save_csv(single.loc[control_protein], 'ptds_control_candidates.csv')
protein=single.loc[~control_protein].copy()
assert protein.Pert_Name2.fillna('').astype(str).str.strip().eq('').all(), 'single 行存在第二药物，请核对'
save_csv(protein,'ptds_mcf7_single_drug_samples.csv')
adata=ad.read_h5ad(RAW/'MCF7_chem_ctrl_adata.h5ad')
assert {'pert_type','condition'} <= set(adata.obs.columns), '作者 obs 字段变化；请查看下方审计'
display(adata.obs.head())
print('表达矩阵:',adata.shape,'pert_type:',adata.obs.pert_type.value_counts().to_dict())
assert adata.n_vars==978, '预期为 978 个 landmark 基因；实际不符，请核对版本'
for cell_col in ['cell_line','cell_id','cell_iname','cell_mfc_name']:
    if cell_col in adata.obs:
        assert adata.obs[cell_col].dropna().map(cell_key).eq(TARGET_CELL).all(), '出现其他细胞系'
obs=adata.obs.copy()
obs['source_obs_id']=obs.index.astype(str)
obs['rna_row_index']=np.arange(len(obs))
save_csv(obs,'rna_obs_inventory.csv')
comp=pd.read_csv(RAW/'processed_CMAP_compound_info.csv')
assert {'pert_id','canonical_smiles'} <= set(comp.columns)
audit={'obs_columns':list(obs.columns),'var_columns':list(adata.var.columns),
       'shape':list(adata.shape),'pert_types':obs.pert_type.astype(str).value_counts().to_dict(),
       'paper_condition_only':{'time_h':24,'dose_uM':10},
       'note':'paper_condition_only 来自论文 Methods，不代表逐行元数据字段'}
(OUT/'schema_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')


## 4. 药物结构匹配：避免 BRD 编号、盐型和药名拼写差异
作者 `condition` 通常是 BRD 编号，不能直接与药名比较。用作者化合物表映射结构；PTDS 药物名通过 PubChem 查询。

RDKit 标准化后取母体、去电荷，使用**完整 InChIKey（保留立体化学）**匹配。此处明确允许同一活性母体的盐型/水合物跨来源匹配；不同衍生物不自动合并。
PubChem 无结果或多候选时保存到复核表，**不使用模糊名称匹配自动通过**。

需要手动指定时，在 BASE 下创建 `drug_overrides.csv`，列为 `ptds_drug,pubchem_cid,reason`，填写核验后的 CID 和依据，再重跑本节及后续。原始查询和结果缓存到 Drive。


In [ ]:
from urllib.parse import quote
def parent_key(mol):
    if mol is None: return None
    try:
        mol=rdMolStandardize.Cleanup(mol)
        mol=rdMolStandardize.FragmentParent(mol)
        mol=rdMolStandardize.Uncharger().uncharge(mol)
        return Chem.MolToInchiKey(mol) or None
    except Exception: return None

cache_path=RAW/'pubchem_cache.json'
cache=json.loads(cache_path.read_text()) if cache_path.exists() else {}
override_path=BASE/'drug_overrides.csv'
overrides=pd.read_csv(override_path) if override_path.exists() else pd.DataFrame(columns=['ptds_drug','pubchem_cid','reason'])
assert not overrides.ptds_drug.duplicated().any(), '同一药物只能有一个人工 CID'
def resolve_drug(name):
    query=re.sub(r'\s*\(=#\d+\)', '', str(name)).strip()
    manual=overrides.loc[overrides.ptds_drug.eq(name)]
    if len(manual):
        row=manual.iloc[0]
        assert pd.notna(row.reason) and str(row.reason).strip(), '人工映射必须记录依据'
        endpoint=f'cid/{int(row.pubchem_cid)}'
        basis='manual_cid: '+str(row.reason)
    else:
        endpoint='name/'+quote(query,safe='')
        basis='pubchem_name_unique_candidate'
    url=f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{endpoint}/property/InChI,InChIKey/JSON'
    try:
        if url in cache: props=cache[url]
        else:
            r=session.get(url,timeout=(30,60))
            r.raise_for_status()
            props=r.json()['PropertyTable']['Properties']
            cache[url]=props
            cache_path.write_text(json.dumps(cache,ensure_ascii=False,indent=2),encoding='utf-8')
            time.sleep(0.25)
        if len(props)!=1: return {'ptds_drug':name,'query':query,'status':'ambiguous','basis':basis,'parent_inchikey':None}
        p=props[0]
        key=parent_key(Chem.MolFromInchi(p['InChI']))
        return {'ptds_drug':name,'query':query,'pubchem_cid':p['CID'],'pubchem_inchikey':p['InChIKey'],
                'parent_inchikey':key,'status':'resolved' if key else 'structure_failed','basis':basis,'query_url':url}
    except Exception as e:
        return {'ptds_drug':name,'query':query,'status':'lookup_failed','basis':basis,'parent_inchikey':None,'error':str(e)}

drugmap=pd.DataFrame([resolve_drug(n) for n in sorted(protein.Pert_Name1.unique())])
save_csv(drugmap,'ptds_drug_resolution.csv')
save_csv(drugmap.loc[drugmap.status.ne('resolved')],'drug_resolution_review.csv')
comp['parent_inchikey']=comp.canonical_smiles.map(lambda x: parent_key(Chem.MolFromSmiles(x)) if isinstance(x,str) else None)
save_csv(comp,'cmap_compound_structure_map.csv')
display(drugmap.reindex(columns=['ptds_drug','pubchem_cid','status','parent_inchikey']))
print('待复核药物数:',int(drugmap.status.ne('resolved').sum()))


## 5. 联合匹配并导出
输出保留每条 RNA 签名，不按剂量/时间平均，也不制造 RNA 行与蛋白重复的一一对应。
`matched_pairs.csv` 每行是一个 PTDS 药名与一个 RNA BRD 药物的匹配；可能存在同母体多个 BRD 条目。
`matched_rna_with_controls.h5ad` 包含匹配的药物行及该作者 MCF7 文件中的对照，新增 `alignment_role` 区分。原始矩阵数值不变。


In [ ]:
chem=obs.loc[obs.pert_type.astype(str).eq('trt_cp')].copy()
chem['rna_pert_id']=chem['condition'].astype(str)
assert len(chem), '没有化学扰动记录'
id_overlap=chem.rna_pert_id.isin(comp.pert_id)
save_csv(chem.loc[~id_overlap], 'rna_unmapped_brd_rows.csv')
assert id_overlap.any(), 'condition 无法与作者 pert_id 对上；请检查 schema_audit，不猜测编号格式'
chem=chem.merge(comp[['pert_id','parent_inchikey']].drop_duplicates(),left_on='rna_pert_id',right_on='pert_id',how='left')
valid_comp=comp.dropna(subset=['parent_inchikey'])
assert valid_comp.groupby('pert_id').parent_inchikey.nunique().le(1).all(), '同 BRD 对应冲突结构'
available=chem.dropna(subset=['parent_inchikey'])[['rna_pert_id','parent_inchikey']].drop_duplicates()
pairs=drugmap.loc[drugmap.status.eq('resolved')].merge(available,on='parent_inchikey',how='inner')
pairs['cell_key']=TARGET_CELL
pairs['cellosaurus_id']='CVCL_0031'
pairs['match_rule']='same_cell_and_standardized_parent_full_inchikey; dose_time_not_filtered'
rna_counts=chem.groupby('rna_pert_id').rna_row_index.nunique()
protein_counts=protein.groupby('Pert_Name1').Sample_ID.nunique()
pairs['n_rna_signatures']=pairs.rna_pert_id.map(rna_counts)
pairs['n_protein_samples']=pairs.ptds_drug.map(protein_counts)
save_csv(pairs,'matched_pairs.csv')
matched_ids=set(pairs.rna_pert_id)
matched_drugs=set(pairs.ptds_drug)
save_csv(protein.loc[protein.Pert_Name1.isin(matched_drugs)],'matched_ptds_samples.csv')
unmatched=drugmap.loc[~drugmap.ptds_drug.isin(matched_drugs)].copy()
unmatched['reason']=np.where(unmatched.status.eq('resolved'),'no_same_parent_in_author_MCF7_chemical_subset','requires_resolution_review')
save_csv(unmatched,'unmatched_ptds_drugs.csv')
treated=obs.pert_type.astype(str).eq('trt_cp') & obs.condition.astype(str).isin(matched_ids)
# 作者 loader 的 control_key 是 ctrl；严格要求非药物行同时有 ctrl 条件，不包含 trt_xpr。
controls=obs.condition.astype(str).eq('ctrl') & obs.pert_type.astype(str).eq('ctl_vehicle.cns')
save_csv(obs.loc[treated],'matched_rna_obs.csv')
save_csv(obs.loc[controls],'rna_controls.csv')
if treated.any():
    assert controls.any(), '有匹配处理但未识别作者对照；请检查 pert_type / condition，不能把基因扰动当对照'
    subset=adata[(treated|controls).to_numpy()].copy()
    subset.obs['alignment_role']=np.where(subset.obs.pert_type.astype(str).eq('trt_cp'),'matched_drug','author_control')
    subset.uns['alignment']={'source_record':str(RECORD_ID),'source_cell':'MCF7',
        'match_key':'cell + standardized parent full InChIKey','dose_filtered':False,'time_filtered':False,
        'matrix_interpretation':'author processed LINCS2020 Level 5 landmark signature; not raw counts'}
    subset.write_h5ad(OUT/'matched_rna_with_controls.h5ad',compression='gzip')
    # 同时导出处理行的矩阵，行号对应 matched_rna_obs.csv 的 rna_row_index；保持顺序。
    from scipy import sparse
    x=adata[treated.to_numpy()].X
    x=x.toarray() if sparse.issparse(x) else np.asarray(x)
    expression=pd.DataFrame(x,columns=adata.var_names.astype(str))
    expression.insert(0,'rna_row_index',obs.loc[treated,'rna_row_index'].to_numpy())
    save_csv(expression,'matched_rna_expression.csv')
    np.testing.assert_allclose(x,subset[subset.obs.alignment_role.eq('matched_drug').to_numpy()].X.toarray()
        if sparse.issparse(subset.X) else subset[subset.obs.alignment_role.eq('matched_drug').to_numpy()].X)
else:
    # 重跑得到零交集时，防止旧表达输出被误认为本轮结果。
    for name in ['matched_rna_with_controls.h5ad','matched_rna_expression.csv']:
        (OUT/name).unlink(missing_ok=True)
    print('本轮无交集：已输出复核表，不生成表达矩阵。')
summary={'ptds_cell':'MCF7','ptds_unique_drugs':int(protein.Pert_Name1.nunique()),
         'matched_ptds_drugs':len(matched_drugs),'matched_rna_brd_ids':len(matched_ids),
         'matched_rna_signatures':int(treated.sum()),'author_control_rows':int(controls.sum()),
         'unresolved_ptds_drugs':int(drugmap.status.ne('resolved').sum()),
         'unmapped_rna_brd_rows':int((~id_overlap).sum()),
         'ptds_sample_info_sha256':sha256(PTDS_XLSX),'run_utc':datetime.now(timezone.utc).isoformat(),
         'rdkit_version':rdBase.rdkitVersion,'python':platform.python_version(),
         'dose_filter':False,'time_filter':False,'matrix_transformation':'none'}
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(pairs)
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('结果目录:',OUT)


## 结果解读与下一步
- 先检查 `matched_pairs.csv` 的药物及结构依据，再看 `drug_resolution_review.csv` 和 `unmatched_ptds_drugs.csv`。PubChem 无结果不等于 RNA 中没有该药物。
- `matched_rna_expression.csv` 与 `matched_rna_obs.csv` 用 `rna_row_index` 对齐，不能用不保证唯一的原始 obs 名称拼接。
- `matched_ptds_samples.csv` 保留 PTDS 的剂量、时间、BioRep 和 Sample_ID，用于以后从蛋白矩阵取列。当前 notebook 不读取蛋白强度文件、不做基因—蛋白映射、不训练模型。
- RNA 对照来自作者文件；PTDS 的 `no` 行另存，双方对照定义和批次条件仍需分别核实。跨实验同条件是弱配对，不能当作同一生物样本。
- 当前只覆盖作者发布的 MCF7 子集，不能将无匹配解读为整个 LINCS 没有该药。若需要扩展覆盖，再单独处理 LINCS 原始版本。
- 2026-10-16 本地真实数据验证（RDKit 2026.09.1）：63 个 PTDS 名称中 62 个解析成功；匹配 20 种 PTDS 药物、21 个 BRD 条目、200 条 RNA 签名及 1 条作者对照。`Verteporfin` 的 PubChem 查询返回 404，需人工复核，因此不宣称这是完整交集。验证包含 CSV/h5ad 导出、h5ad 重新读取和行索引对齐；盐型归并及立体异构体区分检查通过。
- 挂载 Drive、Colab 安装及完整运行仍需以你自己的 Colab 记录为准。建议运行后保存 notebook 副本及关键输出；本 notebook 的执行输出保持为空。
